# Consumer Groups: Rebalancing & Offset Commits

Run this notebook cell-by-cell against the live cluster spawned from the topic page. See `concept.md` for the what/why background.

**This topic runs no Spark job.** The driver container this Jupyter kernel runs in has no Docker CLI/socket access, so the `kafka-consumer-groups.sh --describe` steps below (marked "host terminal") must be run from your own machine's terminal, not from a notebook cell -- the same constraint `kafka-architecture-kraft` documents. Everything else -- starting/stopping/killing consumer-group members, producing the backlog, reading the results -- runs from notebook cells via `kafka-python` and a small subprocess helper, `driver/playbook/consumer_group.py`.

**Why members are separate OS processes, not notebook threads.** Scaling a consumer group and simulating a crash both need genuinely independent group members: a rebalance has to see real joins/leaves, and a "kill mid-batch" has to be an uncatchable, real crash (SIGKILL) so the coordinator's background heartbeat thread actually dies with it -- an in-kernel Python thread can't be killed that cleanly. Each "member" below (`m1`, `m2`, ... and later `manual-1`, `auto-1`) is its own OS process, started via `consumer_group.start()` and stopped via `consumer_group.stop()` (graceful) or `consumer_group.crash()` (SIGKILL).

In [ ]:
import ast
import re
import sys
import time

sys.path.insert(0, "/workspace")
from driver.playbook import consumer_group
from kafka import KafkaProducer

BOOTSTRAP_SERVERS = ["kafka-1:9092", "kafka-2:9092", "kafka-3:9092"]
BOOTSTRAP_CSV = ",".join(BOOTSTRAP_SERVERS)

producer = KafkaProducer(bootstrap_servers=BOOTSTRAP_SERVERS)
print("Connected to the cluster:", producer.bootstrap_connected())
producer.close()

In [ ]:
# Small helpers used by every section below: start a member and poll its
# collected stdout log for the events we're waiting on. consumer_group.start()
# spawns its own daemon drain thread internally, so it's already draining
# the moment it returns -- no separate drain mechanism needed here.

logs = {}     # label -> deque of stdout lines
members = {}  # label -> subprocess.Popen


def start_member(label, topic, group, **kwargs):
    proc, log = consumer_group.start(group=group, label=label, topic=topic, bootstrap=BOOTSTRAP_CSV, **kwargs)
    logs[label] = log
    members[label] = proc
    return proc


def assigned_count(label):
    return sum(1 for line in logs.get(label, []) if " ASSIGNED " in line)


def latest_assignment(label):
    for line in reversed(logs.get(label, [])):
        if " ASSIGNED " in line:
            return ast.literal_eval(line.split("ASSIGNED", 1)[1].strip())
    return None


def wait_for_new_assignment(label, since_count, timeout=25):
    deadline = time.time() + timeout
    while time.time() < deadline:
        if assigned_count(label) > since_count:
            return latest_assignment(label)
        time.sleep(0.5)
    raise TimeoutError(f"{label} did not receive a new assignment within {timeout}s")


def processed_count(label):
    return sum(1 for line in logs.get(label, []) if " PROCESSED " in line)


def wait_for_processed_count(label, count, timeout=25):
    deadline = time.time() + timeout
    while time.time() < deadline:
        if processed_count(label) >= count:
            return processed_count(label)
        time.sleep(0.2)
    raise TimeoutError(f"{label} did not process {count} message(s) within {timeout}s")


def processed_offsets(label):
    offsets = []
    for line in logs.get(label, []):
        if " PROCESSED " in line:
            offsets.append(int(re.search(r"offset=(\d+)", line).group(1)))
    return offsets

## 1. Produce a backlog before any consumer starts

`GROUP_TOPIC` auto-creates with 3 partitions on first send (the broker's fixed `KAFKA_NUM_PARTITIONS=3`, same mechanism the other Kafka topics rely on) -- so **N = 3** for every "partition ceiling" step below. Sending a backlog *before* starting any consumer means the first `--describe` actually shows nonzero lag, not an already-drained group.

In [ ]:
GROUP_TOPIC = "consumer-groups-demo"
GROUP = "cg-demo"
BACKLOG_SIZE = 90  # deliberately more than a single member can drain instantly, so lag is visible

producer = KafkaProducer(bootstrap_servers=BOOTSTRAP_SERVERS)
for i in range(BACKLOG_SIZE):
    producer.send(GROUP_TOPIC, value=f"backlog-event-{i}".encode())
producer.flush()
producer.close()
print(f"Produced {BACKLOG_SIZE} messages to {GROUP_TOPIC!r} -- nothing has consumed them yet.")

## 2. Fewer members than partitions (1 member, N=3 partitions)

Start the group's first member. It should be assigned all 3 partitions -- with only one consumer, there's nothing else to hand the other partitions to.

In [ ]:
start_member("m1", GROUP_TOPIC, GROUP, process_delay=0.3)
m1_assignment = wait_for_new_assignment("m1", since_count=0)
print("m1 assigned partitions:", m1_assignment)
assert m1_assignment == [0, 1, 2], f"expected m1 to own all 3 partitions, got {m1_assignment}"

print()
print("Host terminal (this driver container has no Docker CLI/socket access):")
print(f"  docker exec spark-kafka-1 /opt/kafka/bin/kafka-consumer-groups.sh "
      f"--bootstrap-server localhost:9092 --describe --group {GROUP}")

Run the printed command in your own terminal **now**, while `m1` is still draining the 90-message backlog (`process_delay=0.3` gives you roughly 27s). You should see one row per partition, all under the same `CONSUMER-ID`, and a nonzero `LAG` column total -- the group's total lag, per the acceptance criteria.

## 3. Members == partitions (scale to 3 members)

Start two more members in the *same* group. This triggers a rebalance: the coordinator revokes `m1`'s assignment and hands out a fresh one across all 3 members. With exactly N=3 members for N=3 partitions, each should end up with **exactly one**.

In [ ]:
m1_prior = assigned_count("m1")
start_member("m2", GROUP_TOPIC, GROUP, process_delay=0.3)
start_member("m3", GROUP_TOPIC, GROUP, process_delay=0.3)

m1_assignment = wait_for_new_assignment("m1", m1_prior)
m2_assignment = wait_for_new_assignment("m2", since_count=0)
m3_assignment = wait_for_new_assignment("m3", since_count=0)

print("m1:", m1_assignment, " m2:", m2_assignment, " m3:", m3_assignment)

for label, assignment in (("m1", m1_assignment), ("m2", m2_assignment), ("m3", m3_assignment)):
    assert len(assignment) == 1, f"expected {label} to own exactly one partition, got {assignment}"
all_partitions = sorted(m1_assignment + m2_assignment + m3_assignment)
assert all_partitions == [0, 1, 2], f"expected the 3 partitions split with none missing/doubled, got {all_partitions}"

print()
print("Host terminal:")
print(f"  docker exec spark-kafka-1 /opt/kafka/bin/kafka-consumer-groups.sh "
      f"--bootstrap-server localhost:9092 --describe --group {GROUP}")

Re-run the printed command. You should now see 3 rows, each with a **different** `CONSUMER-ID` -- the 1:1 mapping between members and partitions. This is the partition-count ceiling made concrete: parallelism topped out at exactly 3, matching the topic's partition count.

## 4. Members > partitions (scale to 4 members)

Add a 4th member to the same group. There are still only 3 partitions to hand out -- the 4th member should be assigned **nothing**.

In [ ]:
prior_counts = {label: assigned_count(label) for label in ("m1", "m2", "m3")}
start_member("m4", GROUP_TOPIC, GROUP, process_delay=0.3)

m1_assignment = wait_for_new_assignment("m1", prior_counts["m1"])
m2_assignment = wait_for_new_assignment("m2", prior_counts["m2"])
m3_assignment = wait_for_new_assignment("m3", prior_counts["m3"])
m4_assignment = wait_for_new_assignment("m4", since_count=0)

print("m1:", m1_assignment, " m2:", m2_assignment, " m3:", m3_assignment, " m4 (excess):", m4_assignment)

for label, assignment in (("m1", m1_assignment), ("m2", m2_assignment), ("m3", m3_assignment)):
    assert len(assignment) == 1, f"expected {label} to still own exactly one partition, got {assignment}"
assert m4_assignment == [], f"expected the 4th (excess) member to sit idle with zero partitions, got {m4_assignment}"

print()
print("Host terminal:")
print(f"  docker exec spark-kafka-1 /opt/kafka/bin/kafka-consumer-groups.sh "
      f"--bootstrap-server localhost:9092 --describe --group {GROUP}")

Re-run the printed command once more. `m4` shows up in the group's member list but with no partitions assigned -- `concept.md`'s "Why it matters" section explains why: partition count is a hard ceiling, and a 4th consumer past that ceiling buys nothing. Adding partitions to the topic (not more consumers) is the only way past N=3.

Clean up this section's 4 members before moving on, so they don't linger in the group for the next section's `--describe` output.

In [ ]:
for label in ("m1", "m2", "m3", "m4"):
    consumer_group.stop(members[label])
print("Stopped m1-m4.")

## 5. Manual commit: killed mid-batch, restarted

A single manual-commit consumer (`enable.auto.commit=False`) processes messages one at a time, calling `commit()` only *after* each message's simulated work finishes. We SIGKILL it partway through a message's work -- a real, uncatchable crash, not a graceful shutdown -- then start a fresh consumer in the same group and see exactly where it resumes.

In [ ]:
MANUAL_TOPIC = "consumer-groups-crash-manual"
MANUAL_GROUP = "cg-crash-manual"
N_MANUAL = 8

producer = KafkaProducer(bootstrap_servers=BOOTSTRAP_SERVERS)
for i in range(N_MANUAL):
    # One fixed key -> every message lands on the same partition, so offsets
    # are a simple, deterministic 0..N_MANUAL-1 sequence to reason about.
    producer.send(MANUAL_TOPIC, key=b"crash-demo-key", value=f"manual-event-{i}".encode())
producer.flush()
producer.close()

start_member("manual-1", MANUAL_TOPIC, MANUAL_GROUP, commit_mode="manual", process_delay=0.4)
wait_for_new_assignment("manual-1", since_count=0)

wait_for_processed_count("manual-1", 3)   # let it fully process+commit 3 messages
time.sleep(0.15)                          # ...then crash it mid-work on the 4th (still inside its 0.4s "work" sleep)
consumer_group.crash(members["manual-1"])

before_crash = processed_offsets("manual-1")
print("manual-1 processed (and committed) these offsets before being SIGKILLed:", before_crash)
print("The next offset (whatever message was mid-work when killed) was never committed.")

In [ ]:
remaining = N_MANUAL - len(before_crash)
start_member("manual-2", MANUAL_TOPIC, MANUAL_GROUP, commit_mode="manual", process_delay=0.05,
             max_messages=remaining)
# Same pattern as Section 6's cell below: wait for the expected processed
# count via the drained log, then stop() (SIGTERM -> 5s timeout -> SIGKILL
# fallback) instead of Popen.wait() on natural process exit. consumer.close()
# can block indefinitely inside kafka-python's maybe_leave_group() (issue #74,
# Bug A) even after manual-2's own work (process + commit) is already fully
# done -- stop() is guaranteed to terminate the process either way.
wait_for_processed_count("manual-2", remaining)
consumer_group.stop(members["manual-2"])

after_restart = processed_offsets("manual-2")
print("manual-2 (fresh consumer, same group) processed these offsets after restart:", after_restart)

assert after_restart[0] == before_crash[-1] + 1, (
    "expected the restarted consumer to resume from the very next offset after the last COMMITTED one "
    "-- reprocessing whatever was in flight when manual-1 crashed"
)
assert sorted(before_crash + after_restart) == list(range(N_MANUAL)), (
    "manual commit must never silently lose a message -- every offset 0..N_MANUAL-1 must appear exactly "
    "once across the two runs (the crash-boundary message may be the one reprocessed, but nothing before "
    "it repeats and nothing after it is skipped)"
)
print("No message lost. At most one message (the one in flight at crash time) was reprocessed.")

## 6. Auto commit: same crash timing, contrasted

Same shape of demo, `enable.auto.commit=True` this time. The whole backlog is fetched in a single `poll()` call, so the consumer's internal *position* advances to the end of that batch immediately -- **before** this process has finished acting on any of those messages. If the auto-commit timer fires in that gap and the process is then killed, the committed offset has moved past work that was never finished, and that work is never redelivered.

In [ ]:
AUTO_TOPIC = "consumer-groups-crash-auto"
AUTO_GROUP = "cg-crash-auto"
N_AUTO = 8
AUTO_COMMIT_INTERVAL_MS = 300  # short on purpose, so the timer reliably fires during this short demo

producer = KafkaProducer(bootstrap_servers=BOOTSTRAP_SERVERS)
for i in range(N_AUTO):
    producer.send(AUTO_TOPIC, key=b"crash-demo-key", value=f"auto-event-{i}".encode())
producer.flush()
producer.close()

# batch_size deliberately SMALLER than N_AUTO (issue #74, Bug B): with
# batch_size >= N_AUTO the entire backlog is returned by the FIRST poll(),
# so position() jumps straight to the log end before any processing starts --
# every run then loses the WHOLE backlog once the auto-commit timer fires
# once, rather than the partial loss this demo is meant to illustrate
# (concept.md's "why it matters"). A small batch_size makes position()
# advance in bounded steps instead: kafka-python only checks its auto-commit
# deadline inside poll() (see member.py's _consume_loop), and with
# process_delay=0.4s > AUTO_COMMIT_INTERVAL_MS=0.3s, the very first
# post-message "tick" poll() commits the position established by the FIRST
# (smaller) batch, well before a second tick's poll() advances position any
# further. Crashing inside that ~0.4s window (after the first commit lands,
# before the second one can fire) reliably commits AUTO_BATCH_SIZE messages'
# worth of position while only 1 has actually finished processing --
# partial, not total, loss.
AUTO_BATCH_SIZE = 3
start_member(
    "auto-1", AUTO_TOPIC, AUTO_GROUP, commit_mode="auto",
    auto_commit_interval_ms=AUTO_COMMIT_INTERVAL_MS, process_delay=0.4, batch_size=AUTO_BATCH_SIZE,
)
wait_for_new_assignment("auto-1", since_count=0)

wait_for_processed_count("auto-1", 1)   # first message processed -- the first post-message tick poll() should
                                         # have just committed the first (smaller) batch's advanced position
time.sleep(0.3)                         # crash before the SECOND tick (due at ~0.4s after the first message
                                         # finished) can commit an even-further-advanced position
consumer_group.crash(members["auto-1"])

before_crash_auto = processed_offsets("auto-1")
print("auto-1 fully processed these offsets before being SIGKILLed:", before_crash_auto)
print("But the auto-commit already committed a batch's worth of position, not just these.")

In [ ]:
start_member("auto-2", AUTO_TOPIC, AUTO_GROUP, commit_mode="auto",
             auto_commit_interval_ms=AUTO_COMMIT_INTERVAL_MS, process_delay=0.05, max_messages=1)
try:
    wait_for_processed_count("auto-2", 1, timeout=15)
except TimeoutError:
    # Defensive fallback (issue #74, Bug B): if timing jitter still let
    # auto-1's auto-commit advance past the ENTIRE backlog despite the
    # retiming above, there is nothing left in the topic for auto-2 to ever
    # consume and this wait would hang forever. Fail with a clear, actionable
    # message instead of hanging the notebook.
    consumer_group.stop(members["auto-2"])
    raise RuntimeError(
        "auto-2 never processed a message -- auto-1's auto-commit likely advanced past the entire "
        "backlog despite the retiming (timing jitter). Re-run this section from the cell above."
    ) from None
consumer_group.stop(members["auto-2"])

after_restart_auto = processed_offsets("auto-2")
print("auto-2 (fresh consumer, same group) resumed at offset:", after_restart_auto[0])

lost = [o for o in range(N_AUTO) if o not in before_crash_auto and o < after_restart_auto[0]]
print("Messages never processed by anyone, and never redelivered (silently lost):", lost)

assert after_restart_auto[0] > before_crash_auto[-1] + 1, (
    "expected auto-1's auto-commit to have advanced PAST some unfinished work -- if this fails, the "
    "auto-commit timer didn't fire before the crash; rerun this section (or shorten AUTO_COMMIT_INTERVAL_MS)"
)
assert lost, "expected at least one message to be silently lost -- the whole point of the auto-commit contrast"
print()
print("Contrast with Section 5: manual commit reprocessed at most one in-flight message and lost nothing.")
print("Auto commit here silently skipped", len(lost), "message(s) -- the 'weaker at-most-once-on-crash' behavior.")

## 7. What this demonstrates

- **Partition count is a hard ceiling on parallelism.** 1 member owns all 3 partitions; 3 members own exactly 1 each; a 4th member owns 0. Adding consumers past the partition count doesn't add throughput -- it adds an idle process.
- **Manual commit gives at-least-once, never silent loss.** A crash mid-batch reprocesses at most the one message that was in flight; every message eventually gets processed.
- **Auto commit trades that guarantee for simplicity.** Because it commits the consumer's *position* (advanced the instant a batch is fetched) rather than "work actually finished," a crash between a batch being fetched and the app finishing it can silently drop messages -- they're never redelivered, because Kafka only redelivers from the committed offset forward.

Read `concept.md`'s "Why it matters" section for when that trade-off is (and isn't) acceptable.